# Citadel — Budget enforcement validation

Validates D2/D3/D4: precedence walk, cache TTL behaviour, 80% warn headers, 100% hard 429, `Retry-After` value, and `adminOverride` bypass.

**Stub.** Live execution requires the env gates listed below.

## Validation gate

| # | Required | Variable |
|---|----------|----------|
| 1 | APIM gateway URL | `APIM_GATEWAY_URL` |
| 2 | Claude deployment name | `CLAUDE_DEPLOYMENT_NAME` |
| 3 | JWT for a test user with a known low budget (~1k tokens) | `TEST_LOW_BUDGET_JWT` |
| 4 | JWT for a test user with `adminOverride=true` | `TEST_OVERRIDE_JWT` |
| 5 | OID of low-budget user (for direct Cosmos verification) | `TEST_LOW_BUDGET_OID` |
| 6 | Cosmos account name | `COSMOS_ACCOUNT` |

In [ ]:
import os
REQUIRED = ['APIM_GATEWAY_URL','CLAUDE_DEPLOYMENT_NAME','TEST_LOW_BUDGET_JWT','TEST_OVERRIDE_JWT','TEST_LOW_BUDGET_OID','COSMOS_ACCOUNT']
missing = [k for k in REQUIRED if not os.getenv(k)]
if missing: print('SKIP — missing:', missing)
else: print('OK')

In [ ]:
# Test 1: low-budget user — drive consumption past 80% and verify warning headers appear.
import os, requests
URL = f"{os.environ['APIM_GATEWAY_URL']}/anthropic/v1/messages"
HEADERS = {'Authorization': f"Bearer {os.environ['TEST_LOW_BUDGET_JWT']}", 'Content-Type': 'application/json'}
BODY = {'model': os.environ['CLAUDE_DEPLOYMENT_NAME'], 'max_tokens': 256,
        'messages': [{'role':'user','content':'Write a 200-word essay about token budgets.'}]}

saw_warn = False
for i in range(20):
    r = requests.post(URL, headers=HEADERS, json=BODY)
    pct = r.headers.get('x-citadel-budget-pct')
    rem = r.headers.get('x-citadel-budget-remaining')
    print(f'iter={i} status={r.status_code} pct={pct} remaining={rem}')
    if pct and pct not in ('override',) and float(pct) >= 0.80:
        saw_warn = True
    if r.status_code == 429:
        # Verify Retry-After is sensible: seconds until next month UTC
        ra = int(r.headers['Retry-After'])
        assert 0 < ra <= 32*24*3600, f'Retry-After looks wrong: {ra}'
        print('Hit 429 with Retry-After =', ra, 'seconds')
        break
assert saw_warn, 'never saw the 80% warning header'
assert r.status_code == 429, 'never hit the 100% hard cap'

In [ ]:
# Test 2: adminOverride user — should never see 429 or warning headers.
HEADERS_OVR = {'Authorization': f"Bearer {os.environ['TEST_OVERRIDE_JWT']}", 'Content-Type': 'application/json'}
for i in range(5):
    r = requests.post(URL, headers=HEADERS_OVR, json=BODY)
    assert r.status_code == 200, f'override user got {r.status_code}: {r.text}'
    assert r.headers.get('x-citadel-budget-pct', 'override') == 'override'
    print('override iter', i, 'OK')

In [ ]:
# Test 3: cache cross-user isolation (Risk #4 regression test).
# Send same model from both users; verify the override user is not denied even after low-budget user is throttled.
r_low = requests.post(URL, headers=HEADERS, json=BODY)
r_ovr = requests.post(URL, headers=HEADERS_OVR, json=BODY)
assert r_low.status_code in (200, 429)
assert r_ovr.status_code == 200, 'override user blocked — likely cache cross-user bleed'
print('cache isolation OK')

In [ ]:
# Test 4: Cosmos ai-usage-monthly doc was actually written (Logic App ingestion).
# Optional: requires az login + Cosmos data permission on this notebook host.
import subprocess, json, datetime
month = datetime.datetime.utcnow().strftime('%Y-%m')
oid = os.environ['TEST_LOW_BUDGET_OID']
out = subprocess.check_output([
    'az','cosmosdb','sql','query',
    '--account-name', os.environ['COSMOS_ACCOUNT'],
    '--database-name','ai-usage-db',
    '--container-name','ai-usage-monthly',
    '--query-text', f"SELECT * FROM c WHERE c.oid = '{oid}' AND c.month = '{month}'"
])
docs = json.loads(out)
assert len(docs) >= 1, 'No ai-usage-monthly doc — Logic App patch may not be wired.'
print('Cosmos doc seen:', docs[0])